In [ ]:
!apt-get install tesseract-ocr -y
!pip install pytesseract pdf2image pillow pandas google-generativeai

In [ ]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from google.colab import files
import pandas as pd
import json
import google.generativeai as genai

In [4]:
genai.configure(api_key="")

model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

In [29]:
def load_images(path):

    if path.lower().endswith(".pdf"):
        return convert_from_path(path)
    else:
        return [Image.open(path)]

images = load_images(file_path)

In [ ]:
full_text = ""

for img in images:
    text = pytesseract.image_to_string(img)
    full_text += text + "\n"

print(full_text[:2000])   # preview

In [31]:
prompt = f"""
You are an expert healthcare claim data extraction system.

Your task is to extract structured information from OCR text of a US healthcare claim form.

IMPORTANT RULES:
1. Extract ONLY the requested fields.
2. DO NOT guess or invent values.
3. If a field is missing, return null.
4. Output MUST be valid JSON.
5. Do NOT include explanations, comments, or markdown.
6. Normalize values when possible:
   - Dates → YYYY-MM-DD
   - Amounts → numbers only (no $ or commas)
7. If multiple values exist, choose the most relevant claim-level value.
8. In diagnosis code , if it not an alphanumeric number then remove the decimal

FIELDS TO EXTRACT:
- patient_name
- member_id
- provider_name
- date_of_service
- diagnosis_code
- procedure_code
- charge_amount

Return EXACTLY this JSON structure:

{{
  "patient_name": null,
  "member_id": null,
  "provider_name": null,
  "date_of_service": null,
  "diagnosis_code": null,
  "procedure_code": null,
  "charge_amount": null
}}

Claim Text:
--------------------
{full_text}
--------------------
"""

In [ ]:
response = model.generate_content(prompt)

structured_output = response.text

print(structured_output)

In [ ]:
clean = structured_output.replace("```json","").replace("```","")

data = json.loads(clean)

print(data)

In [ ]:
print("Raw CPT from LLM:", data.get("procedure_code"))

In [40]:
def normalize_cpt(code):

    if code is None:
        return None

    code = str(code).lower().strip()

    replacements = {
        "o": "0",
        "g": "9",
        "t": "1",
        "l": "1",
        "i": "1",
        "s": "5",
        "b": "8"
    }

    corrected = ""

    for ch in code:
        if ch.isdigit():
            corrected += ch
        elif ch in replacements:
            corrected += replacements[ch]

    corrected = "".join(c for c in corrected if c.isdigit())

    if len(corrected) == 5:
        return corrected

    return None

In [ ]:
data["procedure_code"] = normalize_cpt(
    data.get("procedure_code")
)

print("Corrected CPT:", data["procedure_code"])

In [13]:
def excel_safe(value):
    if pd.isna(value):
        return value
    return f'="{value}"'

In [14]:
df = pd.DataFrame([data])

protect_cols = [
    "procedure_code",
    "diagnosis_code",
    "member_id"
]

for col in protect_cols:
    if col in df.columns:
        df[col] = df[col].apply(excel_safe)

df.to_csv("claim_output.csv", index=False)

In [ ]:
df=pd.read_csv("claim_output.csv")
df

In [18]:
files.download("claim_output.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>